In [22]:
import pandas as pd
import numpy as np
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

import torch

In [23]:
MODEL_NAME = "distilbert-base-uncased"
DATA_PATH = "data/guardrail_dataset_augmented.csv"
OUTPUT_DIR = "model"

In [24]:
MAX_LENGTH = 256
NUM_LABELS = 4

In [25]:
label2id = {
    "VALID":0,
    "OUT_OF_SCOPE":1,
    "JAILBREAK_PI_ADV":2,
    "PHI_PII_FLAG":3
}

id2label = {v: k for k, v in label2id.items()}

In [26]:
df = pd.read_csv("data/guardrail_dataset_augmented.csv")
df["labels"] = df["LABEL"].map(label2id)

In [27]:
df.head()

,QUERY,LABEL,labels
0,What is the average BMI of patients in the dat...,VALID,0
1,What is the average body mass index of patient...,VALID,0
2,What is the medium body mass index of patients...,VALID,0
3,What is the average BMI of patient in the data...,VALID,0
4,What is the average age of patients with chron...,VALID,0


In [28]:
df['LABEL'].value_counts()

LABEL
JAILBREAK_PI_ADV    540
VALID               300
PHI_PII_FLAG        220
OUT_OF_SCOPE        132
Name: count, dtype: int64

In [31]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["LABEL"],
    random_state=42,
)



In [32]:
train_df['LABEL'].value_counts()

LABEL
JAILBREAK_PI_ADV    432
VALID               240
PHI_PII_FLAG        176
OUT_OF_SCOPE        105
Name: count, dtype: int64

In [33]:
val_df['LABEL'].value_counts()

LABEL
JAILBREAK_PI_ADV    108
VALID                60
PHI_PII_FLAG         44
OUT_OF_SCOPE         27
Name: count, dtype: int64

In [34]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)


In [35]:
def tokenize(example):
    return tokenizer(
        example["QUERY"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

In [36]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(tokenize)
val_dataset = val_dataset.map(tokenize)

Map: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 239/239 [00:00<00:00, 12360.22 examples/s]


In [37]:
train_dataset = train_dataset.remove_columns(["QUERY", "__index_level_0__","LABEL"])
val_dataset = val_dataset.remove_columns(["QUERY", "__index_level_0__","LABEL"])


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [38]:
print("Loading model...")

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

Loading model...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [39]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }

In [40]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    logging_dir="logs",
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
)

In [41]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

/var/folders/2x/5h669t095bx0w3lgry1c1wrm0000gn/T/ipykernel_59257/3740418003.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [42]:
trainer.train()

/opt/homebrew/Caskroom/miniforge/base/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.540000,0.377263,0.895397,0.858853,0.909029,0.895397
2,0.130800,0.096650,0.970711,0.970452,0.970297,0.970711
3,0.040600,0.060188,0.974895,0.974895,0.974895,0.974895
4,0.012000,0.047239,0.987448,0.987690,0.988703,0.987448
5,0.008300,0.041830,0.991632,0.991742,0.992209,0.991632
6,0.006400,0.042572,0.991632,0.991742,0.992209,0.991632
7,0.005600,0.041874,0.991632,0.991742,0.992209,0.991632
8,0.005600,0.042099,0.991632,0.991742,0.992209,0.991632


/opt/homebrew/Caskroom/miniforge/base/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/homebrew/Caskroom/miniforge/base/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/homebrew/Caskroom/miniforge/base/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/homebrew/Caskroom/miniforge/base/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=480, training_loss=0.14821452586911618, metrics={'train_runtime': 160.6982, 'train_samples_per_second': 47.443, 'train_steps_per_second': 2.987, 'total_flos': 73601201275056.0, 'train_loss': 0.14821452586911618, 'epoch': 8.0})

In [43]:
metrics = trainer.evaluate()

print("\nEvaluation Metrics:")
print(metrics)


/opt/homebrew/Caskroom/miniforge/base/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Evaluation Metrics:
{'eval_loss': 0.04183030501008034, 'eval_accuracy': 0.9916317991631799, 'eval_f1': 0.9917421275787521, 'eval_precision': 0.9922089164622709, 'eval_recall': 0.9916317991631799, 'eval_runtime': 0.9405, 'eval_samples_per_second': 254.122, 'eval_steps_per_second': 15.949, 'epoch': 8.0}


In [44]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\nModel saved to: {OUTPUT_DIR}")


Model saved to: model


In [ ]:
#########